# Fine-Tune Mistral-7B-v0.3 with LoRA on Docker Q&A

**Task #5** of the weekend plan: the first fine-tuning run (LoRA r=16). We load the **base** Mistral-7B in 4-bit, attach LoRA adapters, train for 3 epochs on `data/train/train_1k.jsonl`, and save the adapter so Sunday's eval can score it.

**Runtime budget**: ~1–2 hours on a free Colab T4.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. You've accepted Mistral-7B's license on HuggingFace (it's gated).
3. `HF_TOKEN` is set in Colab Secrets (🔑 left sidebar). `WANDB_API_KEY` is optional.
4. The latest commits are pushed to the GitHub repo cloned below — Colab pulls a fresh copy each run.

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [ ]:
!nvidia-smi

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [ ]:
!git clone https://github.com/<your-username>/LLM_Finetune.git
%cd LLM_Finetune

## 3. Install training dependencies

Compared to the baseline notebook we additionally need:
- **`unsloth`** — fast LoRA training kernels (~2× faster than vanilla on T4). Pulls compatible `trl`, `peft`, `accelerate`, `bitsandbytes` as transitive deps.
- Our `requirements.txt` for `rouge_score`, `google-genai`, `python-dotenv`, etc.

**Why we do NOT `pip install -U trl peft accelerate bitsandbytes`**: Unsloth's compiled cache is generated against a *specific* TRL version. If you upgrade TRL after installing Unsloth, you can break Unsloth's monkey-patches with errors like:

```
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'push_to_hub_token'
```

Let Unsloth pin its own deps — don't `-U` them.

Install takes ~2 minutes.

In [ ]:
%%capture
# Install unsloth + its compatible trl/peft/accelerate/bitsandbytes pins.
# Note: NO `-U` on trl/peft/accelerate/bitsandbytes — that would break
# Unsloth's compiled cache. unsloth's own setup.py is the source of truth.
!pip install -q unsloth datasets
!pip install -q -r requirements.txt

## 4. Load secrets

We read tokens from the Colab Secrets panel into env vars. `HF_TOKEN` is required to download the gated Mistral weights. `WANDB_API_KEY` is optional — if present, we enable Weights & Biases logging on the training run.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    USE_WANDB = True
except Exception:
    USE_WANDB = False

print(f"HF_TOKEN:       {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"WANDB_API_KEY:  {'set' if USE_WANDB else 'not set (W&B disabled)'}")

## 5. Run the fine-tune

This will:
1. Download the Unsloth pre-quantized Mistral-7B 4-bit weights (~4 GB, ~1 minute).
2. Attach LoRA adapters (r=16 by default — see `configs/base.yaml`).
3. Format each row of `data/train/train_1k.jsonl` into Mistral's `[INST]...[/INST]` chat format.
4. Train for 3 epochs (~375 optimizer steps at effective batch size 8).
5. Save the LoRA adapter (~80 MB) to `models/adapters/r16/`.
6. Save a training summary to `results/runs/r16/`.

**Loss expectations**: starts around 1.5–2.0, should drop to ~0.5–0.8 by end of training. If it stays >1.5 the whole run, something is off (check the chat-template example printed at the top — the `[INST]...[/INST]` markers should be visible).

In [ ]:
# Defensive cleanup: Unsloth caches a compiled SFTTrainer wrapper to
# unsloth_compiled_cache/ on first run. If a previous failed run left a
# stale cache (compiled against a different TRL version), Unsloth will
# reuse it and crash. Nuking it forces a fresh compile against the
# currently-installed TRL.
!rm -rf unsloth_compiled_cache

wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/04_train.py --config configs/base.yaml {wandb_flag}

## 6. Inspect the saved adapter

After training we have two output locations:
- `models/adapters/r16/` — the LoRA weights + tokenizer config. This is what `scripts/05_eval_finetuned.py` loads on Sunday.
- `results/runs/r16/` — `config.yaml` (frozen snapshot of hyperparameters) + `train_summary.json`.

The adapter folder should be ~80 MB total — a tiny fraction of the 14 GB base model.

In [ ]:
!ls -lh models/adapters/r16/
print()
!cat results/runs/r16/train_summary.json

## 7. Smoke test — generate one answer from the fine-tuned model

Quick visual check that the adapter actually changed the model's behaviour. We pick an eval question and generate an answer with the freshly-trained model.

**This is NOT the formal eval** (that's Sunday's job, scripts/05). It's a sanity check that:
1. The adapter loaded correctly,
2. The model produces Docker-flavoured answers,
3. The output looks better than the baseline rambles we saw earlier.

In [ ]:
import sys
sys.path.insert(0, ".")

# Quick sanity check that the adapter actually got saved.
!ls -la models/adapters/r16/

from unsloth import FastLanguageModel
from peft import PeftModel
from src.data.format_prompts import format_chat, ensure_chat_template

# Inference is a TWO-STEP load:
#   1) Load the same 4-bit base model we trained from. Unsloth has the
#      pre-quantized version cached from training, so this is fast.
#   2) Layer our trained LoRA adapter ON TOP via PeftModel.from_pretrained.
#
# Why two steps: `models/adapters/r16/` contains only the ~80 MB LoRA delta
# (adapter_config.json + adapter_model.safetensors) — NOT a full standalone
# model. Asking FastLanguageModel.from_pretrained to load it directly fails
# with "No config file found" because there's no full-model config.json.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
ensure_chat_template(tokenizer)

model = PeftModel.from_pretrained(model, "models/adapters/r16")

# Switch into Unsloth's optimized inference mode (disables dropout, etc.)
FastLanguageModel.for_inference(model)

question = "What is the difference between COPY and ADD in a Dockerfile?"
prompt   = format_chat(question, tokenizer)
inputs   = tokenizer(prompt, return_tensors="pt").to("cuda")

out_ids  = model.generate(**inputs, max_new_tokens=256, do_sample=False)
response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", question)
print("A:", response)

## 8. Persist the adapter so Sunday's eval can find it

`/content/` on Colab is **wiped when the runtime disconnects**. If you plan to come back tomorrow for Sunday's eval, you need to save the adapter somewhere persistent.

**Option A — Save to Google Drive (recommended).** Run the next cell. Sunday's notebook re-mounts the same Drive folder.

**Option B — Download a zip locally** (use if you don't want to mount Drive). Drag the resulting zip out of the file browser, then re-upload it tomorrow before Sunday's eval.

In [ ]:
# Option A: copy adapter + results to Google Drive
from google.colab import drive
drive.mount("/content/drive")

import shutil, pathlib
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Mirror models/adapters/r16/ and results/runs/r16/ into Drive
shutil.copytree("models/adapters/r16", DRIVE_ROOT / "models/adapters/r16", dirs_exist_ok=True)
shutil.copytree("results/runs/r16",    DRIVE_ROOT / "results/runs/r16",    dirs_exist_ok=True)
print(f"Adapter + run dir saved to {DRIVE_ROOT}")

In [ ]:
# Option B (alternative): zip + download to local machine
# Uncomment to use.
#
# import shutil
# shutil.make_archive("r16_adapter", "zip", "models/adapters/r16")
# from google.colab import files
# files.download("r16_adapter.zip")

## 9. Commit the small artifacts back to git

The adapter itself is gitignored (`models/` excluded by `.gitignore`). But the **frozen config + training summary** in `results/runs/r16/` are tiny and reproducible — those go to git so the README's results table has an audit trail.

**You will git-push from your local machine after this notebook.** That is, copy `results/runs/r16/config.yaml` and `results/runs/r16/train_summary.json` to your local repo (via Drive or download), then `git add` + `git commit` + `git push` locally. We deliberately avoid pushing from Colab to keep the project's git history clean (no `colab` author commits).